# 03 — Cohort definition

This is where I narrow the ~100 demo stays into the analytic cohort I'll actually model on. The point of doing this in a separate notebook is that the inclusion/exclusion criteria are decisions that deserve to be explicit and replayable — both for me reading my own code later, and for anyone trying to reproduce.

## Criteria

1. **Adults**: `anchor_age >= 18`.
2. **First ICU stay per patient**: avoids correlation between rows of the same patient.
3. **Length of stay >= 24h**: I'm predicting from the first 24h of the stay, so anyone who left or died before 24h can't have a complete feature window. This is a standard simplification; it biases the cohort against the earliest deaths, which I note here and revisit later.
4. **No missing key fields** for the outcome and core demographics.

The resulting cohort is saved to disk so the feature and model notebooks can start from a clean snapshot.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path("../data")
HOSP = DATA_DIR / "hosp"
ICU = DATA_DIR / "icu"
DERIVED = DATA_DIR / "derived"
DERIVED.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

## Load the full stay-level table

In [ ]:
query = f"""
SELECT
    i.subject_id,
    i.hadm_id,
    i.stay_id,
    i.intime,
    i.outtime,
    i.los,
    i.first_careunit,
    p.gender,
    p.anchor_age,
    a.race,
    a.insurance,
    a.marital_status,
    a.hospital_expire_flag
FROM '{ICU / 'icustays.csv.gz'}' AS i
LEFT JOIN '{HOSP / 'patients.csv.gz'}' AS p
    ON i.subject_id = p.subject_id
LEFT JOIN '{HOSP / 'admissions.csv.gz'}' AS a
    ON i.hadm_id = a.hadm_id
"""

stays = con.execute(query).df()
print(f"initial stays: {len(stays)}")
print(f"initial patients: {stays['subject_id'].nunique()}")

## Apply criteria step-by-step

In [ ]:
def report(df, label):
    print(f"{label:<45} stays={len(df):>4}  patients={df['subject_id'].nunique():>4}")

report(stays, "initial")

# 1. Adults
cohort = stays[stays["anchor_age"] >= 18].copy()
report(cohort, "after adult filter (anchor_age >= 18)")

# 2. First ICU stay per patient
cohort = (
    cohort.sort_values(["subject_id", "intime"])
          .drop_duplicates(subset="subject_id", keep="first")
)
report(cohort, "after first ICU stay per patient")

# 3. LOS >= 24h (los is in days)
cohort = cohort[cohort["los"] >= 1.0].copy()
report(cohort, "after LOS >= 24h")

# 4. Drop rows with missing key fields
required = ["hospital_expire_flag", "gender", "anchor_age", "intime"]
cohort = cohort.dropna(subset=required).copy()
report(cohort, "after dropping missing key fields")

## Cohort summary

In [ ]:
print(f"Final cohort size: {len(cohort)} stays / {cohort['subject_id'].nunique()} patients")
print(f"In-hospital mortality rate: {cohort['hospital_expire_flag'].mean():.1%}")
print()
print("Gender:")
print(cohort["gender"].value_counts())
print()
print("Insurance:")
print(cohort["insurance"].value_counts())

In [ ]:
cohort.describe(include="all").T

## Save the cohort

Saving to `data/derived/cohort.parquet` — gitignored like the rest of `data/`. The feature and model notebooks load this file as their starting point.

In [ ]:
out_path = DERIVED / "cohort.parquet"
cohort.to_parquet(out_path, index=False)
print(f"saved to {out_path}")
print(f"file size: {out_path.stat().st_size / 1024:.1f} KB")

## Takeaways

- The cohort is small after the LOS >= 24h filter — expected given the demo's size.
- Mortality rate is worth comparing with the raw rate from notebook 02 to confirm the filtering didn't introduce a major selection effect on the outcome.
- The cohort file is the single source of truth for the next steps. Anything downstream that needs demographics or the outcome reads from this file.

Next (notebook 04): extract first-24h features — vitals from `chartevents`, key labs from `labevents` — and build the feature matrix.